In [ ]:
#!pip install scikit_learn

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, confusion_matrix, precision_score, recall_score, accuracy_score, f1_score

#TASK 3.1 

Build X from age, distance_km, fee_inr, lead_days, prior_no_shows, reminder_sent (convert Yes/No to 1/0 with .map), and the one-hot encoded booking_channel and visit_type (pd.get_dummies(X, columns=["booking_channel","visit_type"])). Do not use patient_id, appointment_id, dates, clinic_id, gender or status as features. Target y = no_show. Split with train_test_split(X, y, test_size=0.2, random_state=42). Report the training and test sizes, the number of no-shows in the test set, and the baseline accuracy of always predicting "Attended". Do not shuffle or re-sort the rows after loading appointments_clean.csv: the exact test-set counts below depend on the row order of that file.

In [ ]:
FILE_PATH_APPOINTMENT = r"C:\Users\CoreX\OneDrive\Desktop\CareLoop_Clinics\Data\appointments_clean.csv"
FILE_PATH_CLINIC = r"C:\Users\CoreX\OneDrive\Desktop\CareLoop_Clinics\Data\clinics_temp.csv"
FILE_PATH = r"C:\Users\CoreX\OneDrive\Desktop\CareLoop_Clinics\Data\test_result.csv"

In [ ]:
df_appt = pd.read_csv(FILE_PATH_APPOINTMENT)
features = ['age', 'distance_km', 'fee_inr', 'booking_channel',
       'visit_type', 'reminder_sent', 'prior_no_shows', 'lead_days']
X = df_appt[features]
y = df_appt['no_show']

In [ ]:
X['reminder_sent'] = X['reminder_sent'].map({'Yes': 1, 'No' : 0})

In [ ]:
X = pd.get_dummies(X, columns=["booking_channel","visit_type"])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state = 42)

In [ ]:
print("The Size of X Train",X_train.shape)
print("The Size of X Test",X_test.shape)
print("No Show in Test Data", len(y_test) - y_test.sum())
print('Accuracy if always predicted as attended', f"{y_test.mean():.2%}")

#Task 3.2

Scale with StandardScaler (scaler = StandardScaler(), X_train_s = scaler.fit_transform(X_train), X_test_s = scaler.transform(X_test) — fit on the training set only), fit LogisticRegression(max_iter=1000), and report accuracy, the confusion matrix, precision, recall, F1 and ROC-AUC on the test set. Print the coefficients (model.coef_[0], one per feature, in the column order of X) next to the feature names and report np.exp(coefficient) for reminder_sent. Remember that the model was fitted on scaled features: to get the effect of a reminder going from No (0) to Yes (1), divide the coefficient by the feature's standard deviation (scaler.scale_) before applying np.exp. State both numbers and explain in one sentence why they differ. Word your conclusion as an association ("in this model, holding the other features constant …"): the data are observational, so it does not prove that reminders cause fewer no-shows.

In [ ]:
scaler = StandardScaler()
scaler.fit(X_train)
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.fit_transform(X_test)

In [ ]:
model_log_reg = LogisticRegression(max_iter=10_000)
model_log_reg.fit(X_train_s, y_train)
y_pred = model_log_reg.predict(X_test_s)
y_prob_lr = model_log_reg.predict_proba(X_test_s)[:,1]

In [ ]:
print("Confusion Matrix : ", "\n", confusion_matrix(y_test, y_pred))

In [ ]:
accuracy_score_lr = accuracy_score(y_test, y_pred)
recall_score_lr = recall_score(y_test, y_pred)
roc_auc_score_lr = roc_auc_score(y_test, y_prob_lr)

print("Precision Score : ", f"{precision_score(y_test, y_pred):.2%}")
print("Recall Score : ", f"{recall_score_lr:.2%}")
print("Accuracy Score : ", f"{accuracy_score_lr:.2%}")
print("F1 Score : ", f"{f1_score(y_test, y_pred):.2%}")
print("ROC_AUC Score : ", f"{roc_auc_score_lr:.2%}")

In [ ]:
for n in range(len(X.columns)):
    print(X.columns[n], ": ", model_log_reg.coef_[0][n])

In [ ]:
df_coef = pd.DataFrame(data = model_log_reg.coef_[0], index=X.columns)
print(df_coef)

In [ ]:
#print(np.exp(df_coef.loc['reminder_sent']/ X['reminder_sent'].std()))
print(np.exp(df_coef.loc['reminder_sent']/ scaler.scale_[3]))


In [ ]:
print("In this model, holding the other features constant if reminder is sent to the customer then there is 3 times more chance that he will show up")

# Task 3.2 
Fit DecisionTreeClassifier(random_state=42) with no depth limit, and again with max_depth=3. For each, report training accuracy and test accuracy, and state in one sentence what the gap shows.

In [ ]:
#with no max_depth limit
model_dt_n_depth = DecisionTreeClassifier(random_state=42)
model_dt_n_depth.fit(X_train, y_train)
#with max_depth as 3
model_dt_3_depth = DecisionTreeClassifier(random_state=42, max_depth=3)
model_dt_3_depth.fit(X_train, y_train)

In [ ]:
y_pred_n_test = model_dt_n_depth.predict(X_test)
y_pred_n_train = model_dt_n_depth.predict(X_train)

y_pred_3_test = model_dt_3_depth.predict(X_test)
y_pred_3_train = model_dt_3_depth.predict(X_train)
y_prob_3_test = model_dt_3_depth.predict_proba(X_test)[:,1]

In [ ]:
#calculating the training and testing accuracy 
print("Accuracy Score for no depth decision tree")
training_n_accuracy = accuracy_score(y_train, y_pred_n_train)
training_3_accuracy = accuracy_score(y_train, y_pred_3_train)
testing_n_accuracy = accuracy_score(y_test, y_pred_n_test)
testing_3_accuracy = accuracy_score(y_test, y_pred_3_test)
recall_score_3_dt = recall_score(y_test, y_pred_3_test) 
roc_auc_score_3_dt = roc_auc_score(y_test, y_prob_3_test)
print(f"Training Accuracy No Max Depth : {training_n_accuracy:.2%}")
print(f"Training Accuracy Max Depth = 3 : {training_3_accuracy:.2%}")
print(f"Testing Accuracy No Max Depth : {testing_n_accuracy:.2%}")
print(f"Testing Accuracy Max Depth = 3: {testing_3_accuracy:.2%}")

In [ ]:
print(f'''
The training accurancy of decision tree with no max depth is {training_n_accuracy:.2%} 
but accuracy on test dataset is {testing_n_accuracy:.2%}
This indicates that model has overfitted. 
after specifiying the max depth parameter to 3, 
training accuracy falls to {training_3_accuracy:.2%} 
with improved test accuracy score of {testing_3_accuracy:.2%}
''')

# Task 3.3 
Fit RandomForestClassifier(n_estimators=100, random_state=42). Report accuracy, recall and ROC-AUC (use predict_proba(X_test)[:, 1] for AUC) and the top three rf.feature_importances_ (the attribute name is given because the course mentioned the idea only briefly). Then build a table with one row each for logistic regression, the depth-3 tree and the random forest (accuracy, recall, ROC-AUC) and say which model looks best and why;

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
model_rf = RandomForestClassifier(n_estimators=100, random_state=42)

In [ ]:
model_rf.fit(X_train, y_train)

In [ ]:
y_pred_rf = model_rf.predict(X_test)
y_prob_rf = model_rf.predict_proba(X_test)[:,1]

In [ ]:
print("With Random Forest Classifier")
test_accuracy_rf = accuracy_score(y_test, y_pred_rf)
recall_score_rf = recall_score(y_test, y_pred_rf)
roc_auc_score_rf = roc_auc_score(y_test, y_prob_rf)
print(f"Accurancy Score : {test_accuracy_rf:.2%}")
print(f"Recall Score : {recall_score_rf:.2%}")
print(f"Accurancy Score : {roc_auc_score_rf:.2%}")

In [ ]:
print("with random forest classifer top 3 features")
df_rf_features = pd.DataFrame(data = {'features' : X_train.columns, 'importance' : model_rf.feature_importances_}).sort_values(by='importance', ascending=False)
print(df_rf_features.head(3))

In [ ]:
#comparison table for 3 models 
df_model_comp = pd.DataFrame(
    data= [
        {'Accuracy': accuracy_score_lr, 'Recall': recall_score_lr, 'ROC_AUC' : roc_auc_score_lr},
        {'Accuracy': testing_3_accuracy, 'Recall': recall_score_3_dt, 'ROC_AUC' : roc_auc_score_3_dt},
        {'Accuracy': test_accuracy_rf, 'Recall': recall_score_rf, 'ROC_AUC' : roc_auc_score_rf}
    ],
    index = ['Logistic Regression', 'Decision Tree', 'Random Forest']
)

In [ ]:
print(df_model_comp)
print('''
As the Logistic Regression Model has 
higest accuracy, recall and precision amongst 3 model which are trained.
Logistic Regression looks best''')

# Task 3.5 
- Run GridSearchCV(RandomForestClassifier(random_state=42), {"n_estimators": [100, 200], "max_depth": [3, 5, 8]}, cv=5, scoring="roc_auc") on the training set; print best_params_ and best_score_ (a cross-validated ROC-AUC). 
- Choose the final model without touching the test set: compute the 5-fold cross-validated ROC-AUC on the training set for logistic regression, the depth-3 tree and the random forest — cross_val_score(model, X_train, y_train, cv=5, scoring="roc_auc").mean() — and compare them with the tuned forest's best_score_. For logistic regression, give cross_val_score the unscaled X_train together with the supplied pipeline below, so that the scaler is re-fitted on each fold's own training part (scaling the whole training set first would let each validation fold influence the scaler). The highest wins. 
- Choose the threshold without touching the test set: get out-of-fold training probabilities for the winner with cross_val_predict(model, X_train, y_train, cv=5, method="predict_proba")[:, 1] (for logistic regression use the pipeline and the unscaled X_train) and compare thresholds 0.5 and 0.35 on precision, recall, F1 and number flagged; pick the threshold with the higher F1 and say why recall matters here (a missed warning costs an empty slot, an extra call costs a minute). 
- Fit the winner on the training set (the pipeline fits its own scaler) and evaluate it once on the test set at your chosen threshold: precision, recall, number flagged and ROC-AUC. 
- Save test_risk_scores.csv with columns appointment_id, no_show, risk_prob, flag (1 if risk_prob ≥ your threshold) for every test row. (To get appointment_id, keep it aside before building X, or use X_test.index to look it up in the loaded DataFrame.) In one sentence, say how a clinic would really choose the threshold: by how many follow-up calls the front desk can make in an evening, against the cost of an empty slot.

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
param_grid = {
    'n_estimators': [100,200],
    'max_depth': [3,5,8],
}
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="roc_auc"
)

In [ ]:
grid_search.fit(X_train, y_train)

In [ ]:
print("For random forest grid search")
print(f"Best Parameters are : {grid_search.best_params_}")
print(f"Best Score is {grid_search.best_score_:.2%}")

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, cross_val_predict

lr_pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])
cv_auc_lr = cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean()
oof = cross_val_predict(lr_pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]
lr_pipe.fit(X_train, y_train)
probs = lr_pipe.predict_proba(X_test)[:, 1]

In [ ]:
cv_auc_dt = cross_val_score(
    DecisionTreeClassifier(max_depth=3),
    X_train, 
    y_train,
    cv=5,
    scoring="roc_auc"
).mean()

In [ ]:
cv_auc_rf = cross_val_score(
    RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    X_train, 
    y_train,
    cv=5,
    scoring="roc_auc"
).mean()

In [ ]:
print(" ROC_AUC Score for three models calculated with cross validation")
print(f"Logistic Regression: {cv_auc_lr:.2%}")
print(f"Decision Tree: {cv_auc_dt:.2%}")
print(f"Random Forest: {cv_auc_rf:.2%}")

In [ ]:
print(f"The highest roc_auc_score is for logistic regression ({cv_auc_lr:.2%}) \n hence selecting logistic regression for next step")

In [ ]:
y_pred_35 = (oof >= 0.35).astype('int')
y_pred_50 = (oof >= 0.50).astype('int')


In [ ]:
print("With default threshold 0.5")
print(f"Precision socre: {precision_score(y_train, y_pred_50):.2%}")
print(f"Recall socre: {recall_score(y_train, y_pred_50):.2%}")
print(f"F1 socre: {f1_score(y_train, y_pred_50):.2%}")
print("\n")
print("With threshold 0.35")
print(f"Precision socre: {precision_score(y_train, y_pred_35):.2%}")
print(f"Recall socre: {recall_score(y_train, y_pred_35):.2%}")
print(f"F1 socre: {f1_score(y_train, y_pred_35):.2%}")

In [ ]:
print(f'''
The recall tells us that out of total positive cases (no show),
How much the model would catch, 
As the cost of a follow up call is very less (assume ₹ 1)
as compare to going doctor's one slot vacant (assume ₹ 1000)
higher recall is alway preferred here. 
The recall with 0.35 threshold is {recall_score(y_train, y_pred_35):.2%}
as compare to 0.5 threshold {recall_score(y_train, y_pred_50):.2%}
so the 0.35 threshold catches almost twice the no shows 
Hence preferred
''')

In [ ]:
y_prob_final_lr  = cross_val_predict(
    estimator=lr_pipe,
    X=X_test,
    y=y_test,
    cv=5,
    method='predict_proba'
)[:,1]

In [ ]:
y_pred_final = (y_prob_final_lr >= 0.35).astype('int')

In [ ]:
print("The performance of Logistic Regression with threshold set to 0.35 ")
print(f"Precision Score : {precision_score(y_test, y_pred_final):.2%}")
print(f"Recall Score : {recall_score(y_test, y_pred_final):.2%}")
print(f"ROC_AUC Score : {roc_auc_score(y_test, y_pred_final):.2%}")
print(f"No of cases flagged", (y_pred_final == 1).sum())

In [ ]:
test_result_df = df_appt.loc[X_test.index,["appointment_id", "no_show"]].copy()

In [ ]:
test_result_df['risk_prob'] = y_prob_final_lr
test_result_df['flag'] = y_pred_final

In [ ]:
import os 
test_result_df.to_csv(FILE_PATH)